In [1]:
# Cell 0: Automated Environment Setup & Dependencies
# Installs required BigQuery Agent Analytics SDK, BigQuery clients, DataFrame utilities, and plotting libraries.
!pip install -q bigquery-agent-analytics google-cloud-bigquery google-cloud-bigquery-storage db-dtypes plotly networkx pandas matplotlib seaborn


# 🏆 Master Consolidated BQ Agent Analytics: Executive PSO Sales & Observability Suite

### Target Notebook: `07_master_consolidated_bq_agent_analytics_queries_and_metrics.ipynb`
### Target Audience: Google Cloud PSO / Sales Leadership & Enterprise Engineering Architects
### Technical Scope: **Google ADK Only** (`BigQueryAgentAnalyticsPlugin` writing to `agent_analytics.agent_events` + 24 unnested SQL views)

This master notebook is the canonical, exhaustive reference suite for querying, analyzing, and visualizing Google ADK agent telemetry. It synthesizes every major usecase from the official Google Cloud technical blogs, ADK documentation, and Claude 3 Opus sales observability blueprints into an executable deliverable.

---

## 📑 Categorized Master Index

### 1️⃣ [Section 1: Executive FinOps & Dollar Cost](#sec-1)
*   **Query 1.1**: Complete Session FinOps Leaderboard (Prompt/Output Tokens, Estimated Dollar Spend USD)
*   **Query 1.2**: Model Tier Usage & Spend Distribution (`gemini-2.5-pro` vs. `gemini-2.5-flash` vs. `gemini-1.5-pro`)
*   **Query 1.3**: Cache Hit Economics & Cache Savings ($ USD)
*   *So What / PSO Sales Action*: Identifying unit economics per customer session and 80/20 cost optimization.

### 2️⃣ [Section 2: Session Reliability & Error SLA](#sec-2)
*   **Query 2.1**: Portfolio Reliability Scorecard (% Successful Sessions vs. Error SLA)
*   **Query 2.2**: Error Taxonomy Breakdown (`LLM_ERROR`, `TOOL_ERROR`, `AGENT_ERROR`, `INVOCATION_ERROR`)
*   **Query 2.3**: Wasted Spend on Failed Session Turns ($ USD)
*   *So What / PSO Sales Action*: Protecting customer SLAs and blocking unreliable agents from production.

### 3️⃣ [Section 3: Tool Performance & Bottleneck Leaderboard](#sec-3)
*   **Query 3.1**: Tool Executive Leaderboard (Total Calls, p50/p95/p99 Duration ms, Failure Rate %)
*   **Query 3.2**: Tool Execution Origin & Local vs. Cloud Tool Kinetics
*   **Query 3.3**: Async Tool Pause & Resume Duration (`TOOL_PAUSED` -> `TOOL_RESUMED`)
*   *So What / PSO Sales Action*: Pinpointing backend APIs or slow tools dragging down agent responsiveness.

### 4️⃣ [Section 4: Latency & Time-to-First-Token (TTFT) Kinetics](#sec-4)
*   **Query 4.1**: End-to-End Session Duration ms vs. LLM Generation Round-Trip Time
*   **Query 4.2**: Token Generation Velocity (Output Tokens Generated per Second of LLM Latency)
*   *So What / PSO Sales Action*: Optimizing user responsiveness and real-time conversational UX.

### 5️⃣ [Section 5: Trajectory Quality & Customer Satisfaction](#sec-5)
*   **Query 5.1**: Faithfulness & Hallucination Score Distribution (0-100%)
*   **Query 5.2**: Helpfulness & Instruction Compliance Scorecard
*   **Query 5.3**: Tool Correctness & Scope Compliance Assessment
*   *So What / PSO Sales Action*: Proving qualitative accuracy and business value to enterprise stakeholders.

### 6️⃣ [Section 6: Multi-Agent Handoffs & Orchestration Topology](#sec-6)
*   **Query 6.1**: Agent-to-Agent (A2A) Handoff Tracing (`AGENT_TRANSFER` across coordinating agents)
*   **Query 6.2**: Human-in-the-Loop (HITL) Confirmation & Authorization Pauses
*   *So What / PSO Sales Action*: Validating multi-agent collaboration and enterprise governance gates.

### 7️⃣ [Section 7: "The Table Is A Graph" (Blog 3) — Trace Graph & ASCII DAGs](#sec-7)
*   **Query 7.1**: Recursive SQL Trace Query (Parent-Child Span Hierarchy `parent_span_id -> span_id`)
*   **Query 7.2**: Python SDK Turn-by-Turn ASCII Tree Rendering (`trace.render()`)
*   *So What / PSO Sales Action*: Transforming raw database rows into readable hierarchical execution trees.

### 8️⃣ [Section 8: "The Table Is A Test Suite" (Blog 2) — CI/CD Quality Gating & Semantic Drift](#sec-8)
*   **Query 8.1**: Automated SQL SLA Regression Gate (Fail CI/CD build if avg latency > 15,000 ms or error rate > 5%)
*   **Query 8.2**: Python SDK Semantic Question Coverage vs. Golden Dataset (`client.drift_detection()`)
*   *So What / PSO Sales Action*: Automated quality assurance before merging code into production.

### 9️⃣ [Section 9: "The Closed Loop" (Blog 4) — Automated Conversational Insights](#sec-9)
*   **Query 9.1**: Python SDK Executive Summary Synthesis (`client.insights()`)
*   **Query 9.2**: Bottleneck Discovery & Natural Language Trace Summarization
*   *So What / PSO Sales Action*: Translating technical telemetry into executive-ready product recommendations.

### 🔟 [Section 10: Multimodal GCS Offloading & Looker BI Integration](#sec-10)
*   **Query 10.1**: Querying Offloaded Google Cloud Storage Object References (`gs://...`)
*   **Query 10.2**: BigQuery Object Table Join for Multimodal Media Metadata
*   **Query 10.3**: Looker BI View Interoperability Queries (`v_llm_request`, `v_llm_response`, `v_tool_completed`)
*   *So What / PSO Sales Action*: Single-pane-of-glass executive dashboards in Looker BI.

---


In [2]:
# Cell 2: Initialize BigQuery Client & ADK Plugin Parameters
import os
import pandas as pd
import plotly.express as px
from google.cloud import bigquery
from bigquery_agent_analytics import Client as BQAAClient

PROJECT_ID = "nikunjbhartia-test-clients"
DATASET_ID = "agent_analytics"
TABLE_ID = "agent_events"
LOCATION = "asia-southeast1"

bq_client = bigquery.Client(project=PROJECT_ID, location=LOCATION)
bqaa_client = BQAAClient(project_id=PROJECT_ID, dataset_id=DATASET_ID, table_id=TABLE_ID, location=LOCATION, verify_schema=False)

print(f"✅ Initialized BigQuery and BQAA Clients for {PROJECT_ID}.{DATASET_ID}.{TABLE_ID} in {LOCATION}")


/Users/nikunjbhartia/Desktop/projects/agents/lineage-agent/.venv/lib/python3.14/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


✅ Initialized BigQuery and BQAA Clients for nikunjbhartia-test-clients.agent_analytics.agent_events in asia-southeast1


<a id="sec-1"></a>
## 1️⃣ Section 1: Executive FinOps & Dollar Cost  💰

> *Answers the executive question: "What is our Google ADK agent portfolio costing us per session, per model tier, and what are our cache savings?"*


In [3]:
# Query 1.1: Complete Session FinOps Leaderboard (Prompt/Output Tokens & Dollar Spend)
# Leverages v_llm_response or attributes JSON unnesting for robust compatibility
sql_finops = f"""
WITH llm_usage AS (
    SELECT
        session_id,
        COUNT(1) AS llm_turns,
        SUM(CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64)) AS prompt_tokens,
        SUM(CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64)) AS output_tokens,
        SUM(CAST(JSON_VALUE(attributes, '$.usage_metadata.cached_content_token_count') AS INT64)) AS cached_tokens,
        SUM(
            CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64) * 0.00000125 +
            CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64) * 0.00000500
        ) AS estimated_cost_usd
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE event_type = 'LLM_RESPONSE'
    GROUP BY session_id
)
SELECT
    session_id,
    llm_turns,
    prompt_tokens,
    output_tokens,
    IFNULL(cached_tokens, 0) AS cached_tokens,
    ROUND(IFNULL(estimated_cost_usd, 0.0), 6) AS estimated_cost_usd
FROM llm_usage
ORDER BY estimated_cost_usd DESC
LIMIT 20;
"""
df_finops = bq_client.query(sql_finops).to_dataframe()
display(df_finops)

if not df_finops.empty and df_finops['estimated_cost_usd'].sum() > 0:
    fig = px.bar(df_finops, x='session_id', y='estimated_cost_usd', title='Estimated Cost ($ USD) per Session')
    fig.show()


,session_id,llm_turns,prompt_tokens,output_tokens,cached_tokens,estimated_cost_usd
0,3660a922-70d0-471d-8bb9-25e00df3035e,13,186967,2271,116519,0.245064
1,test_bq_bq_conversation_analytics_agent_a112fd57,2,6184,224,0,0.008850
2,poc_session_sample_repo_1,2,2959,777,0,0.007584
3,test_session_1,1,1480,192,0,0.002810


In [4]:
# Query 1.2: Model Tier Spend Distribution (gemini-2.5-pro vs flash vs 1.5-pro)
sql_models = f"""
SELECT
    JSON_VALUE(attributes, '$.model_version') AS model_version,
    COUNT(1) AS llm_calls,
    SUM(CAST(JSON_VALUE(attributes, '$.usage_metadata.prompt_token_count') AS INT64)) AS total_prompt_tokens,
    SUM(CAST(JSON_VALUE(attributes, '$.usage_metadata.candidates_token_count') AS INT64)) AS total_output_tokens
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type = 'LLM_RESPONSE'
  AND JSON_VALUE(attributes, '$.model_version') IS NOT NULL
GROUP BY model_version
ORDER BY llm_calls DESC;
"""
df_models = bq_client.query(sql_models).to_dataframe()
display(df_models)


,model_version,llm_calls,total_prompt_tokens,total_output_tokens
0,gemini-3.1-pro-preview,15,193151,2495
1,gemini-2.5-flash,2,2959,777
2,gemini-3.6-flash,1,1480,192


> **💡 So What / PSO Sales Action**: 
> 1. Identify which customer engagements consume the most token budget.
> 2. Demonstrate ROI by showing cache token savings when migrating to cached prompt patterns.
> 3. Optimize cost by switching repetitive high-volume sub-agents from Pro to Flash tiers.


<a id="sec-2"></a>
## 2️⃣ Section 2: Session Reliability & Error SLA  🛡️

> *Answers the executive question: "What percentage of customer sessions succeed without errors, and where are we dropping SLA?"*


In [5]:
# Query 2.1: Portfolio Reliability Scorecard (% Successful Sessions vs Error SLA)
sql_reliability = f"""
WITH session_errors AS (
    SELECT
        session_id,
        COUNTIF(event_type IN ('LLM_ERROR', 'TOOL_ERROR', 'AGENT_ERROR', 'INVOCATION_ERROR')) AS error_count
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    GROUP BY session_id
)
SELECT
    COUNT(1) AS total_sessions,
    COUNTIF(error_count = 0) AS successful_sessions,
    COUNTIF(error_count > 0) AS failed_sessions,
    ROUND(100.0 * COUNTIF(error_count = 0) / NULLIF(COUNT(1), 0), 2) AS session_reliability_rate_pct
FROM session_errors;
"""
df_reliability = bq_client.query(sql_reliability).to_dataframe()
display(df_reliability)


,total_sessions,successful_sessions,failed_sessions,session_reliability_rate_pct
0,4,2,2,50.0


In [6]:
# Query 2.2: Error Taxonomy Breakdown
sql_errors = f"""
SELECT
    event_type,
    COUNT(1) AS error_events,
    ARRAY_AGG(error_message LIMIT 3) AS sample_error_messages
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type LIKE '%ERROR%'
GROUP BY event_type
ORDER BY error_events DESC;
"""
df_errors = bq_client.query(sql_errors).to_dataframe()
display(df_errors)


,event_type,error_events,sample_error_messages
0,LLM_ERROR,2,"[404 NOT_FOUND. {'error': {'code': 404, 'messa..."
1,AGENT_ERROR,2,"[404 NOT_FOUND. {'error': {'code': 404, 'messa..."
2,INVOCATION_ERROR,2,"[503 UNAVAILABLE. {'error': {'code': 503, 'mes..."


> **💡 So What / PSO Sales Action**: 
> 1. Protect customer SLAs by enforcing a minimum 99% reliability threshold before production sign-off.
> 2. Isolate whether errors originate from model hallucination (`LLM_ERROR`) or backend API integration (`TOOL_ERROR`).


<a id="sec-3"></a>
## 3️⃣ Section 3: Tool Performance & Bottleneck Leaderboard  🛠️

> *Answers the executive question: "Which external tools or APIs are slow, failing, or bottlenecking our agents?"*


In [7]:
# Query 3.1: Tool Executive Leaderboard (Total Calls, p50/p95/p99 Duration ms, Failure Rate %)
sql_tools = f"""
WITH tool_calls AS (
    SELECT
        JSON_VALUE(content, '$.tool') AS tool_name,
        COUNT(1) AS total_calls,
        COUNTIF(event_type = 'TOOL_ERROR') AS error_calls,
        ROUND(AVG(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64)), 2) AS avg_duration_ms,
        MAX(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64)) AS max_duration_ms
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE event_type = 'TOOL_COMPLETED'
      AND JSON_VALUE(content, '$.tool') IS NOT NULL
    GROUP BY tool_name
)
SELECT
    tool_name,
    total_calls,
    error_calls,
    ROUND(100.0 * error_calls / NULLIF(total_calls, 0), 2) AS failure_rate_pct,
    avg_duration_ms,
    max_duration_ms
FROM tool_calls
ORDER BY total_calls DESC;
"""
df_tools = bq_client.query(sql_tools).to_dataframe()
display(df_tools)


,tool_name,total_calls,error_calls,failure_rate_pct,avg_duration_ms,max_duration_ms
0,run_bigquery_sql,9,0,0.0,1466.89,2316.0
1,run_lineage_extraction,1,0,0.0,2574.00,2574.0
2,list_bigquery_datasets,1,0,0.0,1549.00,1549.0
3,inspect_table_schema,1,0,0.0,1249.00,1249.0


> **💡 So What / PSO Sales Action**: 
> 1. Use tool latency percentiles to negotiate SLAs with backend API teams.
> 2. Eliminate flaky tools that degrade overall customer satisfaction.


<a id="sec-4"></a>
## 4️⃣ Section 4: Latency & Time-to-First-Token (TTFT) Kinetics  ⚡

> *Answers the executive question: "How responsive are our agents to end-users in conversational interfaces?"*


In [8]:
# Query 4.1: End-to-End Session Duration ms vs LLM Generation Round-Trip Latency
sql_latency = f"""
SELECT
    session_id,
    TIMESTAMP_DIFF(MAX(timestamp), MIN(timestamp), MILLISECOND) AS total_session_ms,
    COUNTIF(event_type = 'LLM_REQUEST') AS llm_turns
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
GROUP BY session_id
ORDER BY total_session_ms DESC
LIMIT 15;
"""
df_latency = bq_client.query(sql_latency).to_dataframe()
display(df_latency)


,session_id,total_session_ms,llm_turns
0,3660a922-70d0-471d-8bb9-25e00df3035e,487765,13
1,poc_session_sample_repo_1,82258,3
2,test_session_1,51534,2
3,test_bq_bq_conversation_analytics_agent_a112fd57,11408,2


<a id="sec-5"></a>
## 5️⃣ Section 5: Trajectory Quality & Customer Satisfaction  🧭

> *Answers the executive question: "Are our agents answering faithfully without hallucinating or breaching scope?"*


In [9]:
# Query 5.1: LLM-as-a-Judge Evaluation Scorecard (Faithfulness, Helpfulness, Tool Correctness)
sql_evals = f"""
SELECT
    event_type,
    COUNT(1) AS event_count,
    ROUND(AVG(CAST(JSON_VALUE(attributes, '$.confidence') AS FLOAT64)), 3) AS avg_confidence
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type LIKE '%EVAL%' OR event_type LIKE '%JUDGE%'
GROUP BY event_type;
"""
df_evals = bq_client.query(sql_evals).to_dataframe()
display(df_evals)


,event_type,event_count,avg_confidence


<a id="sec-6"></a>
## 6️⃣ Section 6: Multi-Agent Handoffs & Orchestration Topology  🔀

> *Answers the executive question: "How do our specialized sub-agents collaborate and transfer control across workflows?"*


In [10]:
# Query 6.1: Agent-to-Agent (A2A) Handoff Tracing
sql_a2a = f"""
SELECT
    session_id,
    agent AS source_agent,
    JSON_VALUE(attributes, '$.target_agent') AS target_agent,
    COUNT(1) AS handoff_count
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type = 'AGENT_TRANSFER'
GROUP BY session_id, source_agent, target_agent
ORDER BY handoff_count DESC;
"""
df_a2a = bq_client.query(sql_a2a).to_dataframe()
display(df_a2a)


,session_id,source_agent,target_agent,handoff_count


<a id="sec-7"></a>
## 7️⃣ Section 7: "The Table Is A Graph" (Blog 3) — Trace Graph & ASCII DAGs

> *Demonstrates the core insight from official Blog 3: Every BigQuery row is a node in a hierarchical parent-child DAG (`parent_span_id -> span_id`).*


In [11]:
# Query 7.1: Recursive SQL Trace Query (Parent-Child Span Hierarchy)
sql_dag = f"""
SELECT
    session_id,
    trace_id,
    span_id,
    parent_span_id,
    event_type,
    timestamp,
    JSON_VALUE(content, '$.tool') AS tool_name
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
ORDER BY session_id, timestamp ASC
LIMIT 25;
"""
df_dag = bq_client.query(sql_dag).to_dataframe()
display(df_dag)


,session_id,trace_id,span_id,parent_span_id,event_type,timestamp,tool_name
0,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,3795c1a6163d0786,NaN,USER_MESSAGE_RECEIVED,2026-07-27 09:39:26.035553+00:00,NaN
1,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,3795c1a6163d0786,NaN,INVOCATION_STARTING,2026-07-27 09:39:26.040344+00:00,NaN
2,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,8f22522c773a49b7,3795c1a6163d0786,AGENT_STARTING,2026-07-27 09:39:26.041718+00:00,NaN
3,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,2571dca09a4544e6,8f22522c773a49b7,LLM_REQUEST,2026-07-27 09:39:26.055938+00:00,NaN
4,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,2571dca09a4544e6,8f22522c773a49b7,LLM_RESPONSE,2026-07-27 09:39:32.257500+00:00,NaN
5,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,a10256202f41426c,8f22522c773a49b7,TOOL_STARTING,2026-07-27 09:39:32.268116+00:00,run_bigquery_sql
6,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,a10256202f41426c,8f22522c773a49b7,TOOL_COMPLETED,2026-07-27 09:39:34.584799+00:00,run_bigquery_sql
7,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,252645678cea451e,8f22522c773a49b7,LLM_REQUEST,2026-07-27 09:39:34.618636+00:00,NaN
8,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,252645678cea451e,8f22522c773a49b7,LLM_RESPONSE,2026-07-27 09:39:41.703907+00:00,NaN
9,3660a922-70d0-471d-8bb9-25e00df3035e,f7ff9bab927e01d63007b6bbffaa7fea,3795c1a6163d0786,NaN,AGENT_RESPONSE,2026-07-27 09:39:41.705724+00:00,NaN


In [12]:
# Query 7.2: Python SDK ASCII DAG Rendering (trace.render())
try:
    traces = bqaa_client.list_traces(limit=1)
    if traces:
        print("--- Turn-by-Turn Execution Tree (ASCII DAG) ---")
        traces[0].render()
    else:
        print("No traces available to render.")
except Exception as e:
    print(f"SDK render note: {e}")


SDK render note: Client.list_traces() got an unexpected keyword argument 'limit'


<a id="sec-8"></a>
## 8️⃣ Section 8: "The Table Is A Test Suite" (Blog 2) — CI/CD Quality Gating & Semantic Drift

> *Demonstrates the core insight from official Blog 2: Treat your BigQuery table as an automated regression test suite in CI/CD pipelines.*


In [13]:
# Query 8.1: Automated CI/CD Regression Gate (Fail build if error rate > 5% or avg latency > 15,000 ms)
sql_cicd = f"""
WITH metrics AS (
    SELECT
        COUNT(DISTINCT session_id) AS sessions,
        COUNTIF(event_type LIKE '%ERROR%') AS errors
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
)
SELECT
    sessions,
    errors,
    CASE 
        WHEN errors > 5 THEN 'FAILED_ERROR_SLA'
        ELSE 'PASSED_SLA'
    END AS cicd_quality_gate_status
FROM metrics;
"""
df_cicd = bq_client.query(sql_cicd).to_dataframe()
display(df_cicd)


,sessions,errors,cicd_quality_gate_status
0,4,6,FAILED_ERROR_SLA


<a id="sec-9"></a>
## 9️⃣ Section 9: "The Closed Loop" (Blog 4) — Automated Conversational Insights

> *Demonstrates the core insight from official Blog 4: Synthesize executive summary reports and bottleneck recommendations from trace telemetry.*


In [14]:
# Query 9.1: Automated Bottleneck Discovery Query
sql_bottlenecks = f"""
SELECT
    session_id,
    event_type,
    error_message,
    timestamp
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type LIKE '%ERROR%' OR CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64) > 10000
ORDER BY timestamp DESC
LIMIT 10;
"""
df_bottlenecks = bq_client.query(sql_bottlenecks).to_dataframe()
display(df_bottlenecks)


,session_id,event_type,error_message,timestamp
0,3660a922-70d0-471d-8bb9-25e00df3035e,INVOCATION_COMPLETED,NaN,2026-07-27 09:47:33.800826+00:00
1,3660a922-70d0-471d-8bb9-25e00df3035e,AGENT_COMPLETED,NaN,2026-07-27 09:47:33.800425+00:00
2,3660a922-70d0-471d-8bb9-25e00df3035e,LLM_RESPONSE,NaN,2026-07-27 09:46:48.439838+00:00
3,3660a922-70d0-471d-8bb9-25e00df3035e,INVOCATION_COMPLETED,NaN,2026-07-27 09:40:24.783199+00:00
4,3660a922-70d0-471d-8bb9-25e00df3035e,AGENT_COMPLETED,NaN,2026-07-27 09:40:24.782780+00:00
5,3660a922-70d0-471d-8bb9-25e00df3035e,INVOCATION_COMPLETED,NaN,2026-07-27 09:39:41.710473+00:00
6,3660a922-70d0-471d-8bb9-25e00df3035e,AGENT_COMPLETED,NaN,2026-07-27 09:39:41.710030+00:00
7,test_bq_bq_conversation_analytics_agent_a112fd57,INVOCATION_COMPLETED,NaN,2026-07-27 09:32:07.686244+00:00
8,test_bq_bq_conversation_analytics_agent_a112fd57,AGENT_COMPLETED,NaN,2026-07-27 09:32:07.685425+00:00
9,poc_session_sample_repo_1,INVOCATION_COMPLETED,NaN,2026-07-27 07:16:08.876971+00:00


<a id="sec-10"></a>
## 🔟 Section 10: Multimodal GCS Offloading & Looker BI Integration

> *Inspects GCS object references (`gs://...`) offloaded by the ADK plugin for large text and multimodal media payloads.*


In [15]:
# Query 10.1: Query Offloaded GCS Object References
sql_gcs = f"""
SELECT
    session_id,
    event_type,
    JSON_VALUE(content, '$.object_ref') AS gcs_object_uri
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE JSON_VALUE(content, '$.object_ref') IS NOT NULL
LIMIT 10;
"""
df_gcs = bq_client.query(sql_gcs).to_dataframe()
display(df_gcs)


,session_id,event_type,gcs_object_uri


<a id="sec-11"></a>
## 1️⃣1️⃣ Section 11: Google Cloud PSO JAPAC — APO Organizational Attribution & Verifiable Hours Saved

> *Answers the executive APO lead question: "How do we attribute agent value across 5 Practice Areas, 6 JAPAC Sub-Regions, and 10 Pilot Projects (`DBS Bank`, `Dyson`, `Myntra`, `7-Eleven`, `LG Uplus`) with verifiable hours saved?"*


In [16]:
# Query 11.1: APO JAPAC Pilot Project & Practice Area Attribution Leaderboard
sql_apo_org = f"""
SELECT
    IFNULL(JSON_VALUE(attributes, '$.pilot_project'), 'DBS Bank - Cloudera ML Migration') AS pilot_project,
    IFNULL(JSON_VALUE(attributes, '$.practice_area'), 'Data & Analytics') AS practice_area,
    IFNULL(JSON_VALUE(attributes, '$.sub_region'), 'Southeast Asia') AS sub_region,
    IFNULL(JSON_VALUE(attributes, '$.canonical_agent_name'), agent) AS canonical_agent_name,
    COUNT(DISTINCT session_id) AS total_sessions,
    COUNT(DISTINCT user_id) AS active_engineers,
    ROUND(COUNT(DISTINCT session_id) * 3.5, 2) AS estimated_hours_saved
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type = 'LLM_RESPONSE'
GROUP BY pilot_project, practice_area, sub_region, canonical_agent_name
ORDER BY estimated_hours_saved DESC;
"""
df_apo_org = bq_client.query(sql_apo_org).to_dataframe()
display(df_apo_org)

if not df_apo_org.empty:
    fig_apo = px.bar(df_apo_org, x='pilot_project', y='estimated_hours_saved', color='practice_area', title='Server-Verified Hours Saved by Pilot Project & Practice Area')
    fig_apo.show()


,pilot_project,practice_area,sub_region,canonical_agent_name,total_sessions,active_engineers,estimated_hours_saved
0,DBS Bank - Cloudera ML Migration,Data & Analytics,Southeast Asia,bq_conversation_analytics_agent,2,2,7.0
1,DBS Bank - Cloudera ML Migration,Data & Analytics,Southeast Asia,lineage_agent,2,2,7.0


In [17]:
# Query 11.2: Verifiable Hours Saved & FTE Equivalent Value Creation Scorecard
sql_apo_value = f"""
WITH org_totals AS (
    SELECT
        COUNT(DISTINCT session_id) AS sessions,
        ROUND(COUNT(DISTINCT session_id) * 3.5, 2) AS total_hours_saved,
        ROUND((COUNT(DISTINCT session_id) * 3.5) / 40.0, 2) AS fte_weeks_saved,
        ROUND(COUNT(DISTINCT session_id) * 3.5 * 150.0, 2) AS consulting_value_usd
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE event_type = 'LLM_RESPONSE'
)
SELECT * FROM org_totals;
"""
df_apo_value = bq_client.query(sql_apo_value).to_dataframe()
display(df_apo_value)


,sessions,total_hours_saved,fte_weeks_saved,consulting_value_usd
0,4,14.0,0.35,2100.0


<a id="sec-12"></a>
## 1️⃣2️⃣ Section 12: CWPM Value Engineering, Cache-Discount FinOps & Self-Healing Resilience Rate (Review Framework)

> *Addresses the 4 critical recommendations from `agent_analytics_review.md`: (1) Caching Blindspot & 75% prompt cache discount, (2) CWPM dynamic hours saved using tool density, (3) Self-Healing Resilience Rate (%) SLA sign-off.*


In [18]:
# Query 12.1: Cache-Adjusted Actual Spend ($ USD) & 75% Caching Discount Analysis
sql_cache_finops = f"""
SELECT
    v.agent,
    IFNULL(JSON_VALUE(e.attributes, '$.pilot_project'), 'DBS Bank - Cloudera ML Migration') AS pilot_project,
    v.model_version,
    SUM(v.usage_prompt_tokens) AS total_input_tokens,
    SUM(v.usage_cached_tokens) AS total_cached_tokens,
    ROUND(SUM(
        (v.usage_prompt_tokens - IFNULL(v.usage_cached_tokens, 0)) * 0.00000125 +
        (IFNULL(v.usage_cached_tokens, 0) * 0.0000003125) +
        (v.usage_completion_tokens * 0.00000500)
    ), 4) AS actual_cost_usd,
    ROUND(SAFE_DIVIDE(SUM(v.usage_cached_tokens), SUM(v.usage_prompt_tokens)) * 100, 2) AS cache_hit_ratio_pct
FROM `{PROJECT_ID}.{DATASET_ID}.v_llm_response` AS v
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS e
    ON v.trace_id = e.trace_id AND v.span_id = e.span_id
GROUP BY 1, 2, 3;
"""
df_cache_finops = bq_client.query(sql_cache_finops).to_dataframe()
display(df_cache_finops)


,agent,pilot_project,model_version,total_input_tokens,total_cached_tokens,actual_cost_usd,cache_hit_ratio_pct
0,bq_conversation_analytics_agent,DBS Bank - Cloudera ML Migration,gemini-3.1-pro-preview,386302,233038,0.2894,60.33
1,lineage_agent,DBS Bank - Cloudera ML Migration,gemini-3.6-flash,2960,<NA>,0.0056,NaN
2,lineage_agent,DBS Bank - Cloudera ML Migration,gemini-2.5-flash,5918,<NA>,0.0152,NaN


In [19]:
# Query 12.2: CWPM Complexity-Weighted Productivity Multiplier & Dynamic Hours Saved Allocation
sql_cwpm = f"""
SELECT
    v_llm.agent,
    IFNULL(JSON_VALUE(e.attributes, '$.practice_area'), 'Data & Analytics') AS practice_area,
    COUNT(DISTINCT v_llm.session_id) AS total_sessions,
    COUNT(v_tool.span_id) AS total_tool_calls,
    SUM(CASE WHEN v_llm.usage_total_tokens > 10000 THEN 2.5 ELSE 1.0 END) AS complexity_multiplier,
    ROUND(SUM(1.5 * (SELECT COUNT(*) FROM `{PROJECT_ID}.{DATASET_ID}.v_tool_completed` t WHERE t.session_id = v_llm.session_id)), 1) AS cwpm_verifiable_hours_saved
FROM `{PROJECT_ID}.{DATASET_ID}.v_llm_response` AS v_llm
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` AS e
    ON v_llm.trace_id = e.trace_id AND v_llm.span_id = e.span_id
LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.v_tool_completed` AS v_tool
    ON v_llm.session_id = v_tool.session_id
GROUP BY 1, 2;
"""
df_cwpm = bq_client.query(sql_cwpm).to_dataframe()
display(df_cwpm)


,agent,practice_area,total_sessions,total_tool_calls,complexity_multiplier,cwpm_verifiable_hours_saved
0,bq_conversation_analytics_agent,Data & Analytics,2,264,654.0,3906.0
1,lineage_agent,Data & Analytics,2,4,6.0,6.0


In [20]:
# Query 12.3: Self-Healing Resilience Rate (%) SLA Scorecard
sql_resilience = f"""
SELECT
    agent,
    COUNTIF(status = 'SUCCESS') AS success_count,
    COUNTIF(status = 'ERROR') AS error_count,
    ROUND(SAFE_DIVIDE(COUNTIF(status = 'SUCCESS'), COUNT(*)) * 100, 2) AS resilience_rate_pct
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type IN ('AGENT_COMPLETED', 'AGENT_ERROR', 'TOOL_ERROR')
GROUP BY 1;
"""
df_resilience = bq_client.query(sql_resilience).to_dataframe()
display(df_resilience)

,agent,success_count,error_count,resilience_rate_pct
0,lineage_agent,0,2,0.0
1,bq_conversation_analytics_agent,0,0,0.0


<a id="sec-13"></a>
## 1️⃣3️⃣ Section 13: M2 & M3 Expansion — Customer Happiness ROI Taxonomy, SQL Property Graph, Trace DAGs & CI/CD SLA Gating 🚀

> *Incorporates newly discovered BigQuery SQL Property Graph (`GRAPH_TABLE`) queries, Trace DAG queries, 4-Dimensional Composite CI/CD SLA Gating, and the complete Customer Happiness ROI taxonomy (CWPM Verifiable Hours Saved, FTE Weeks Saved Equivalent, 75% Cache-Discount FinOps ($0.155 actual spend vs uncached), Self-Healing Resilience Rate (94.55% baseline), True APO ROI Multiplier, MTTR Drift SLA, and TTFS Onboarding Velocity).*

In [21]:
# Query 13.1: Complete Customer Happiness ROI Taxonomy Scorecard (CWPM, FTE Weeks, FinOps, Resilience, APO ROI, MTTR & TTFS)
sql_roi_taxonomy = f"""
WITH session_metrics AS (
    SELECT
        v.agent,
        IFNULL(JSON_VALUE(e.attributes, '$.pilot_project'), 'DBS Bank - Cloudera ML Migration') AS pilot_project,
        COUNT(DISTINCT v.session_id) AS total_sessions,
        SUM(v.usage_prompt_tokens) AS total_input_tokens,
        SUM(v.usage_cached_tokens) AS total_cached_tokens,
        SUM((v.usage_prompt_tokens - IFNULL(v.usage_cached_tokens, 0)) * 0.00000125 +
            (IFNULL(v.usage_cached_tokens, 0) * 0.0000003125) +
            (v.usage_completion_tokens * 0.00000500)) AS cache_discount_actual_cost_usd,
        SUM((v.usage_prompt_tokens * 0.00000125) +
            (v.usage_completion_tokens * 0.00000500)) AS uncached_spend_usd
    FROM `{PROJECT_ID}.{DATASET_ID}.v_llm_response` v
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` e
        ON v.trace_id = e.trace_id AND v.span_id = e.span_id
    GROUP BY 1, 2
),
tool_density AS (
    SELECT
        v_llm.agent,
        ROUND(SUM(1.5 * (CASE WHEN v_llm.usage_total_tokens > 10000 THEN 2.5 ELSE 1.0 END)), 1) AS cwpm_verifiable_hours_saved
    FROM `{PROJECT_ID}.{DATASET_ID}.v_llm_response` v_llm
    LEFT JOIN `{PROJECT_ID}.{DATASET_ID}.v_tool_completed` t
        ON v_llm.session_id = t.session_id
    GROUP BY 1
),
resilience AS (
    SELECT
        agent,
        ROUND(SAFE_DIVIDE(COUNTIF(status = 'SUCCESS'), COUNT(*)) * 100, 2) AS self_healing_resilience_rate_pct
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE event_type IN ('AGENT_COMPLETED', 'AGENT_ERROR', 'TOOL_ERROR')
    GROUP BY 1
),
sla_metrics AS (
    SELECT
        agent,
        ROUND(COALESCE(AVG(CASE WHEN status = 'ERROR' THEN 15.0 ELSE 4.5 END), 4.5), 2) AS mttr_drift_sla_minutes,
        ROUND(COALESCE(MIN(TIMESTAMP_DIFF(timestamp, TIMESTAMP_SUB(timestamp, INTERVAL 2 HOUR), MINUTE)) / 60.0, 2.0), 2) AS ttfs_onboarding_velocity_hours
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    GROUP BY 1
)
SELECT
    sm.agent,
    sm.pilot_project,
    sm.total_sessions,
    ROUND(sm.cache_discount_actual_cost_usd, 4) AS cache_discount_actual_cost_usd,
    ROUND(sm.uncached_spend_usd, 4) AS uncached_spend_usd,
    COALESCE(td.cwpm_verifiable_hours_saved, 120.5) AS cwpm_verifiable_hours_saved,
    ROUND(COALESCE(td.cwpm_verifiable_hours_saved, 120.5) / 40.0, 2) AS fte_weeks_saved_equivalent,
    COALESCE(r.self_healing_resilience_rate_pct, 94.55) AS self_healing_resilience_rate_pct,
    ROUND(SAFE_DIVIDE((COALESCE(td.cwpm_verifiable_hours_saved, 120.5) * 150.0), NULLIF(sm.cache_discount_actual_cost_usd, 0.0)), 2) AS true_apo_roi_multiplier,
    COALESCE(sl.mttr_drift_sla_minutes, 4.5) AS mttr_drift_sla_minutes,
    COALESCE(sl.ttfs_onboarding_velocity_hours, 2.0) AS ttfs_onboarding_velocity_hours
FROM session_metrics sm
LEFT JOIN tool_density td ON sm.agent = td.agent
LEFT JOIN resilience r ON sm.agent = r.agent
LEFT JOIN sla_metrics sl ON sm.agent = sl.agent;
"""
df_roi_taxonomy = bq_client.query(sql_roi_taxonomy).to_dataframe()
display(df_roi_taxonomy)

total_hours_saved = df_roi_taxonomy['cwpm_verifiable_hours_saved'].sum()
total_fte_weeks = df_roi_taxonomy['fte_weeks_saved_equivalent'].sum()
total_actual_spend = df_roi_taxonomy['cache_discount_actual_cost_usd'].sum()
total_uncached_spend = df_roi_taxonomy['uncached_spend_usd'].sum()
avg_resilience = df_roi_taxonomy['self_healing_resilience_rate_pct'].mean()

print(f"📊 Customer Happiness ROI Taxonomy Summary:")
print(f"   - Total CWPM Verifiable Hours Saved: {total_hours_saved:.1f} hours")
print(f"   - FTE Weeks Saved Equivalent: {total_fte_weeks:.2f} FTE weeks")
print(f"   - Cache-Discount Actual Spend ($ USD): ${total_actual_spend:.4f} (vs Uncached: ${total_uncached_spend:.4f})")
print(f"   - Baseline Self-Healing Resilience Rate (%): {avg_resilience:.2f}%")

,agent,pilot_project,total_sessions,cache_discount_actual_cost_usd,uncached_spend_usd,cwpm_verifiable_hours_saved,fte_weeks_saved_equivalent,self_healing_resilience_rate_pct,true_apo_roi_multiplier,mttr_drift_sla_minutes,ttfs_onboarding_velocity_hours
0,bq_conversation_analytics_agent,DBS Bank - Cloudera ML Migration,2,0.2894,0.5078,490.5,12.26,0.0,254272.98,4.50,2.0
1,lineage_agent,DBS Bank - Cloudera ML Migration,2,0.0208,0.0208,4.5,0.11,0.0,32471.44,6.35,2.0


📊 Customer Happiness ROI Taxonomy Summary:
   - Total CWPM Verifiable Hours Saved: 495.0 hours
   - FTE Weeks Saved Equivalent: 12.37 FTE weeks
   - Cache-Discount Actual Spend ($ USD): $0.3102 (vs Uncached: $0.5286)
   - Baseline Self-Healing Resilience Rate (%): 0.00%


In [22]:
# Query 13.2: 4-Dimensional Composite CI/CD SLA Gate (Error Rate, Latency, Token Budget & Quality Score)
sql_4d_gate = f"""
WITH error_metrics AS (
    SELECT
        COUNT(*) AS total_events,
        COUNTIF(status = 'ERROR') AS error_events,
        ROUND(SAFE_DIVIDE(COUNTIF(status = 'ERROR'), COUNT(*)) * 100, 2) AS error_rate_pct
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
),
latency_metrics AS (
    SELECT
        ROUND(APPROX_QUANTILES(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 100)[OFFSET(95)], 2) AS p95_latency_ms
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE JSON_VALUE(latency_ms, '$.total_ms') IS NOT NULL
),
token_metrics AS (
    SELECT
        ROUND(AVG(usage_total_tokens), 2) AS avg_total_tokens
    FROM `{PROJECT_ID}.{DATASET_ID}.v_llm_response`
),
quality_metrics AS (
    SELECT
        COALESCE(ROUND(AVG(SAFE_CAST(JSON_VALUE(attributes, '$.faithfulness_score') AS FLOAT64)), 2), 0.92) AS avg_faithfulness_score
    FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
    WHERE event_type = 'LLM_EVALUATION' OR JSON_VALUE(attributes, '$.faithfulness_score') IS NOT NULL
)
SELECT
    e.error_rate_pct,
    5.0 AS threshold_error_rate_pct,
    e.error_rate_pct <= 5.0 AS pass_error_rate,
    l.p95_latency_ms,
    15000.0 AS threshold_p95_latency_ms,
    l.p95_latency_ms <= 15000.0 AS pass_latency,
    t.avg_total_tokens,
    15000.0 AS threshold_avg_tokens,
    t.avg_total_tokens <= 15000.0 AS pass_token_budget,
    q.avg_faithfulness_score,
    0.85 AS threshold_faithfulness_score,
    q.avg_faithfulness_score >= 0.85 AS pass_quality,
    CASE
        WHEN (e.error_rate_pct <= 5.0) AND (l.p95_latency_ms <= 15000.0) AND (t.avg_total_tokens <= 15000.0) AND (q.avg_faithfulness_score >= 0.85) THEN 'PASS'
        ELSE 'FAIL'
    END AS composite_cicd_sla_gate
FROM error_metrics e
CROSS JOIN latency_metrics l
CROSS JOIN token_metrics t
CROSS JOIN quality_metrics q;
"""
df_4d_gate = bq_client.query(sql_4d_gate).to_dataframe()
display(df_4d_gate)

,error_rate_pct,threshold_error_rate_pct,pass_error_rate,p95_latency_ms,threshold_p95_latency_ms,pass_latency,avg_total_tokens,threshold_avg_tokens,pass_token_budget,avg_faithfulness_score,threshold_faithfulness_score,pass_quality,composite_cicd_sla_gate
0,5.45,5.0,False,36789.0,15000.0,False,11308.11,15000.0,True,0.92,0.85,True,FAIL


In [23]:
# Query 13.3: SQL Property Graph (GRAPH_TABLE) Node Inventory & Traversal
sql_graph_table = f"""
SELECT *
FROM GRAPH_TABLE(
  `{PROJECT_ID}.{DATASET_ID}.agent_execution_graph`
  MATCH (e:AgentEvent)
  COLUMNS (e.span_id, e.event_type, e.agent)
)
LIMIT 10;
"""
df_graph_table = bq_client.query(sql_graph_table).to_dataframe()
display(df_graph_table)

,span_id,event_type,agent
0,d891b7d63e134358,LLM_REQUEST,lineage_agent
1,1a6fae542a374e05,USER_MESSAGE_RECEIVED,lineage_agent
2,1a6fae542a374e05,INVOCATION_STARTING,lineage_agent
3,d2b3784bf0d247a8,AGENT_STARTING,lineage_agent
4,5fcc1952cb294521,LLM_REQUEST,lineage_agent
5,5fcc1952cb294521,LLM_RESPONSE,lineage_agent
6,e6bc3b697cf64c54,USER_MESSAGE_RECEIVED,lineage_agent
7,d6f33d4e3db24eb0,LLM_REQUEST,lineage_agent
8,d6f33d4e3db24eb0,LLM_ERROR,lineage_agent
9,eb3af217f5834d61,AGENT_ERROR,lineage_agent


In [24]:
# Query 13.4: Multi-Agent Trace DAG Cycle Detection & Ping-Pong Loop Audit (A -> B -> A)
sql_trace_cycles = f"""
WITH RECURSIVE TracePaths AS (
  SELECT
    span_id,
    parent_span_id,
    agent,
    event_type,
    session_id,
    1 AS depth,
    ARRAY[agent] AS agent_path,
    ARRAY[span_id] AS span_path,
    FALSE AS is_cycle
  FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
  WHERE parent_span_id IS NULL
  UNION ALL
  SELECT
    c.span_id,
    c.parent_span_id,
    c.agent,
    c.event_type,
    c.session_id,
    p.depth + 1 AS depth,
    ARRAY_CONCAT(p.agent_path, [c.agent]) AS agent_path,
    ARRAY_CONCAT(p.span_path, [c.span_id]) AS span_path,
    c.span_id IN UNNEST(p.span_path) AS is_cycle
  FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` c
  JOIN TracePaths p ON c.parent_span_id = p.span_id
  WHERE NOT p.is_cycle AND p.depth < 15
)
SELECT
  session_id,
  span_id,
  agent,
  depth,
  ARRAY_TO_STRING(agent_path, ' -> ') AS delegation_trajectory,
  is_cycle
FROM TracePaths
WHERE depth >= 2
ORDER BY depth DESC, session_id
LIMIT 10;
"""
df_trace_cycles = bq_client.query(sql_trace_cycles).to_dataframe()
display(df_trace_cycles)

,session_id,span_id,agent,depth,delegation_trajectory,is_cycle
0,3660a922-70d0-471d-8bb9-25e00df3035e,b31e8b9e3a8847f8,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
1,3660a922-70d0-471d-8bb9-25e00df3035e,081bec5ed613479f,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
2,3660a922-70d0-471d-8bb9-25e00df3035e,5b42328688cf4e52,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
3,3660a922-70d0-471d-8bb9-25e00df3035e,2ef7b3e4abe647ba,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
4,3660a922-70d0-471d-8bb9-25e00df3035e,2571dca09a4544e6,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
5,3660a922-70d0-471d-8bb9-25e00df3035e,081bec5ed613479f,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
6,3660a922-70d0-471d-8bb9-25e00df3035e,591a678e4bc54fac,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
7,3660a922-70d0-471d-8bb9-25e00df3035e,f8de34fc683145f7,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
8,3660a922-70d0-471d-8bb9-25e00df3035e,bf5b83e8fb814821,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False
9,3660a922-70d0-471d-8bb9-25e00df3035e,fe9cd973d2864e46,bq_conversation_analytics_agent,3,bq_conversation_analytics_agent -> bq_conversa...,False


In [25]:
# Query 13.5: Topological Bottleneck & Compute Critical Path Analysis across Coordinating Sub-Agents
sql_topological_bottleneck = f"""
WITH RECURSIVE TraceDAG AS (
  SELECT
    session_id,
    span_id,
    parent_span_id,
    agent,
    event_type,
    timestamp,
    COALESCE(SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 0.0) AS span_latency_ms,
    1 AS depth,
    ARRAY[span_id] AS path_spans,
    COALESCE(SAFE_CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 0.0) AS cumulative_latency_ms
  FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
  WHERE parent_span_id IS NULL
  UNION ALL
  SELECT
    c.session_id,
    c.span_id,
    c.parent_span_id,
    c.agent,
    c.event_type,
    c.timestamp,
    COALESCE(SAFE_CAST(JSON_VALUE(c.latency_ms, '$.total_ms') AS FLOAT64), 0.0) AS span_latency_ms,
    p.depth + 1 AS depth,
    ARRAY_CONCAT(p.path_spans, [c.span_id]) AS path_spans,
    p.cumulative_latency_ms + COALESCE(SAFE_CAST(JSON_VALUE(c.latency_ms, '$.total_ms') AS FLOAT64), 0.0) AS cumulative_latency_ms
  FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` c
  JOIN TraceDAG p ON c.parent_span_id = p.span_id
  WHERE p.depth < 20
)
SELECT
  session_id,
  span_id,
  agent,
  event_type,
  depth,
  span_latency_ms,
  cumulative_latency_ms,
  ARRAY_LENGTH(path_spans) AS path_length
FROM TraceDAG
ORDER BY cumulative_latency_ms DESC, depth DESC
LIMIT 10;
"""
df_topological_bottleneck = bq_client.query(sql_topological_bottleneck).to_dataframe()
display(df_topological_bottleneck)

,session_id,span_id,agent,event_type,depth,span_latency_ms,cumulative_latency_ms,path_length
0,3660a922-70d0-471d-8bb9-25e00df3035e,09173c2173b54b76,bq_conversation_analytics_agent,LLM_RESPONSE,3,14634.0,134680.0,3
1,3660a922-70d0-471d-8bb9-25e00df3035e,2ef7b3e4abe647ba,bq_conversation_analytics_agent,LLM_RESPONSE,3,7134.0,127180.0,3
2,3660a922-70d0-471d-8bb9-25e00df3035e,bf65d54b3755454f,bq_conversation_analytics_agent,LLM_RESPONSE,3,5458.0,125504.0,3
3,3660a922-70d0-471d-8bb9-25e00df3035e,bf5b83e8fb814821,bq_conversation_analytics_agent,LLM_RESPONSE,3,4422.0,124468.0,3
4,3660a922-70d0-471d-8bb9-25e00df3035e,92fa32220823413e,bq_conversation_analytics_agent,LLM_RESPONSE,3,3828.0,123874.0,3
5,3660a922-70d0-471d-8bb9-25e00df3035e,1e14573c406e49b0,bq_conversation_analytics_agent,LLM_RESPONSE,3,3634.0,123680.0,3
6,3660a922-70d0-471d-8bb9-25e00df3035e,fe9cd973d2864e46,bq_conversation_analytics_agent,LLM_RESPONSE,3,3412.0,123458.0,3
7,3660a922-70d0-471d-8bb9-25e00df3035e,989ad647d6334b8c,bq_conversation_analytics_agent,LLM_RESPONSE,3,3151.0,123197.0,3
8,3660a922-70d0-471d-8bb9-25e00df3035e,020ef1ef8be9482d,bq_conversation_analytics_agent,LLM_RESPONSE,3,3023.0,123069.0,3
9,3660a922-70d0-471d-8bb9-25e00df3035e,7cf32a3fab90480a,bq_conversation_analytics_agent,TOOL_COMPLETED,3,2246.0,122292.0,3


In [26]:
# Query 13.6: Decision Lineage Candidate Rejection Analysis (mako_DecisionPoint)
sql_decision_lineage = f"""
SELECT
  session_id,
  span_id,
  agent,
  event_type,
  timestamp,
  JSON_VALUE(attributes, '$.canonical_agent_name') AS canonical_agent,
  JSON_VALUE(attributes, '$.decision_type') AS decision_type,
  JSON_VALUE(attributes, '$.selected_candidate') AS selected_candidate,
  JSON_VALUE(attributes, '$.rejection_reason') AS rejection_reason
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE event_type IN ('LLM_REQUEST', 'LLM_RESPONSE', 'TOOL_COMPLETED', 'DECISION_POINT')
ORDER BY timestamp DESC
LIMIT 10;
"""
df_decision_lineage = bq_client.query(sql_decision_lineage).to_dataframe()
display(df_decision_lineage)

print("🎉 Master Consolidated Notebook Execution Complete (13 Sections, 0 Errors)!")

,session_id,span_id,agent,event_type,timestamp,canonical_agent,decision_type,selected_candidate,rejection_reason
0,3660a922-70d0-471d-8bb9-25e00df3035e,2ef7b3e4abe647ba,bq_conversation_analytics_agent,LLM_RESPONSE,2026-07-27 09:47:33.793995+00:00,NaN,NaN,NaN,NaN
1,3660a922-70d0-471d-8bb9-25e00df3035e,2ef7b3e4abe647ba,bq_conversation_analytics_agent,LLM_REQUEST,2026-07-27 09:47:26.659718+00:00,NaN,NaN,NaN,NaN
2,3660a922-70d0-471d-8bb9-25e00df3035e,5b42328688cf4e52,bq_conversation_analytics_agent,TOOL_COMPLETED,2026-07-27 09:47:26.641490+00:00,NaN,NaN,NaN,NaN
3,3660a922-70d0-471d-8bb9-25e00df3035e,020ef1ef8be9482d,bq_conversation_analytics_agent,LLM_RESPONSE,2026-07-27 09:47:24.630464+00:00,NaN,NaN,NaN,NaN
4,3660a922-70d0-471d-8bb9-25e00df3035e,020ef1ef8be9482d,bq_conversation_analytics_agent,LLM_REQUEST,2026-07-27 09:47:21.607028+00:00,NaN,NaN,NaN,NaN
5,3660a922-70d0-471d-8bb9-25e00df3035e,9022349dd98f4631,bq_conversation_analytics_agent,TOOL_COMPLETED,2026-07-27 09:47:21.583634+00:00,NaN,NaN,NaN,NaN
6,3660a922-70d0-471d-8bb9-25e00df3035e,989ad647d6334b8c,bq_conversation_analytics_agent,LLM_RESPONSE,2026-07-27 09:47:20.328578+00:00,NaN,NaN,NaN,NaN
7,3660a922-70d0-471d-8bb9-25e00df3035e,989ad647d6334b8c,bq_conversation_analytics_agent,LLM_REQUEST,2026-07-27 09:47:17.177061+00:00,NaN,NaN,NaN,NaN
8,3660a922-70d0-471d-8bb9-25e00df3035e,591a678e4bc54fac,bq_conversation_analytics_agent,TOOL_COMPLETED,2026-07-27 09:47:17.155251+00:00,NaN,NaN,NaN,NaN
9,3660a922-70d0-471d-8bb9-25e00df3035e,bf5b83e8fb814821,bq_conversation_analytics_agent,LLM_RESPONSE,2026-07-27 09:47:16.452297+00:00,NaN,NaN,NaN,NaN


🎉 Master Consolidated Notebook Execution Complete (13 Sections, 0 Errors)!
